# Phase 2 — Feature Engineering

**Objective:** Craft behavioral features, each with a testable hypothesis about why it predicts churn.

This notebook re-creates the lazy scans from Notebook 1 (cheap — just a query plan),
builds four feature groups, joins them into one table, and **saves the result to
`features.parquet`** so downstream notebooks don't need to recompute from raw data.

| Feature group | Source table | What it captures |
|---|---|---|
| KYC / demographic | kyc.parquet | Geography |
| Transaction behavior & intensity | transactions | Frequency, recency, amounts, bill-pay habits |
| Temporal gap ratios | transactions | Whether usage gaps are widening or shrinking |
| Wallet liquidity | dayend_balance | Balance stability and trend |


In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import polars as pl

IS_KAGGLE = os.path.exists('/kaggle')
base_path = '/media/tasfreak/Study/bkash-presents-nsucec-datathon/public'
USER_COL = "ACCOUNT_ID"

# Re-create lazy frames (no data read yet — same as Notebook 1)
lazy_balances = pl.scan_parquet(f"{base_path}/dayend_balance/balance_2024-*.parquet")
lazy_transactions = pl.scan_parquet(f"{base_path}/transactions/trx_2024-*.parquet").drop("TrxID")
lazy_kyc = pl.scan_parquet(f"{base_path}/kyc.parquet").drop("GENDER")

## Feature set 1 — KYC / demographic features

**Hypothesis:** Geographic region can correlate with churn due to differing local
competition, network coverage, or merchant density. Encoded as a categorical type
so LightGBM treats it as unordered (not ordinal).

In [2]:
# ==============================================================================
# FEATURE SET 1: KYC / DEMOGRAPHIC FEATURES
# ==============================================================================
dataset_cutoff = pl.date(2024, 3, 31)
type_map = {"Customer": 0, "Biller": 1, "Merchant": 2}
kyc_features = (
    lazy_kyc
    .select([
        "ACCOUNT_ID", 
        "ACCOUNT_TYPE",
    ])
    .with_columns([
        # [FEATURE]: ACCOUNT_TYPE
        # ----------------
        # Encodes the account type into a Polars Categorical type.
        # This keeps memory overhead incredibly low and signals tree-based
        # models (like LightGBM) to treat this column as an unordered category.
        pl.col("ACCOUNT_TYPE").replace(type_map).cast(pl.Int64).alias("ACCOUNT_TYPE")
    ])
)

kyc_features.fetch(10)

ACCOUNT_ID,ACCOUNT_TYPE
str,i64
"""CUST00000001""",0
"""CUST00000002""",0
"""CUST00000003""",0
"""CUST00000004""",0
"""CUST00000005""",0
"""CUST00000006""",0
"""CUST00000007""",0
"""CUST00000008""",0
"""CUST00000009""",0


## Feature set 2 — Transaction behavior & intensity metrics

This is the largest feature group. Covers bill-pay habits, frequency, recency,
momentum (is usage speeding up or slowing down), and "whale" importance relative
to the average user.

| Feature | Hypothesis |
|---|---|
| `bill_pay_ratio` | Bill payments anchor a customer to the platform |
| `unique_trx_types_used` | More transaction variety = deeper product adoption |
| `frequency` / `trx_count_last_30D` | Recent activity level signals engagement |
| `recency` | Days since last transaction — classic churn signal |
| `average_gap_days` | Typical rhythm between transactions |
| `p2p_ratio_90d` | High P2P-only usage suggests casual, less sticky use case |
| `trx_trend` | Momentum: is March activity higher or lower than January? |
| `consistency_metric` | Erratic usage patterns vs. stable habitual ones |
| `importance` | Financial weight relative to platform average — flags "whale" accounts |
| `velocity_last_14d` | Share of lifetime activity concentrated in the last 2 weeks |
| `inactivity_multiplier` | How many multiples of their normal gap a user has been dormant |


In [3]:
# ==============================================================================
# FEATURE SET 2: TRANSACTION BEHAVIOR & INTENSITY METRICS
# ==============================================================================
dataset_cutoff = pl.date(2024, 3, 31)
two_weeks_ago  = pl.date(2024, 3, 16)
one_week_ago  = pl.date(2024, 3, 24)
one_month_ago  = pl.date(2024, 2, 29)

# GLOBAL SCALAR CALCULATION:
# Calculates the overall mean transaction volume across your whole user base.
# Used downstream as a baseline denominator to calculate individual user 'importance'.
avg_total_volume_per_user = (
    lazy_transactions
    .group_by("SRC_ACCOUNT")
    .agg(pl.col("TRX_AMT").sum().alias("user_total_volume"))
    .select(pl.col("user_total_volume").mean())
    .collect()
    .item()
)

trx_features = (
    lazy_transactions
    # Step 1: Pre-convert string timestamps to datetime objects for accurate chronological sorting
    .with_columns(pl.col("TRX_DATETIME").str.to_datetime("%Y-%m-%d %H:%M:%S"))
    .sort("TRX_DATETIME")
    .group_by("SRC_ACCOUNT")
    .agg([
        # --- BILL PAY FEATURES ---
        # ----------------------------------------------------------------------
        # [FEATURE]: bill_pay_count_last_30D
        # Total number of utility/bill payments made by the user.
        pl.col("TRX_AMT")
        .filter(pl.col("TRX_TYPE").str.contains("BillPay") & (pl.col("TRX_DATETIME")>one_month_ago))
        .len()
        .fill_null(0)
        .alias("bill_pay_count_last_30D"),

        # [FEATURE]: bill_pay_count_last_14D
        # Total number of utility/bill payments made by the user.
        pl.col("TRX_AMT")
        .filter(pl.col("TRX_TYPE").str.contains("BillPay") & (pl.col("TRX_DATETIME")>two_weeks_ago))
        .len()
        .fill_null(0)
        .alias("bill_pay_count_last_14D"),

        # # [FEATURE]: bill_pay_count_last_7D
        # # Total number of utility/bill payments made by the user.
        # pl.col("TRX_AMT")
        # .filter(pl.col("TRX_TYPE").str.contains("BillPay") & (pl.col("TRX_DATETIME")>one_week_ago))
        # .len()
        # .fill_null(0)
        # .alias("bill_pay_count_last_7D"),

        # # [FEATURE]: merchant_count_last_30D
        # # Total number of merchant payments made by the user.
        # pl.col("TRX_AMT")
        # .filter(pl.col("TRX_TYPE").str.contains("MerchantPay") & (pl.col("TRX_DATETIME")>one_month_ago))
        # .len()
        # .fill_null(0)
        # .alias("merchant_count_last_30D"),

        # # [FEATURE]: merchant_count_last_14D
        # # Total number of merchant payments made by the user.
        # pl.col("TRX_AMT")
        # .filter(pl.col("TRX_TYPE").str.contains("MerchantPay") & (pl.col("TRX_DATETIME")>two_weeks_ago))
        # .len()
        # .fill_null(0)
        # .alias("merchant_count_last_14D"),

        # # [FEATURE]: merchant_count_last_7D
        # # Total number of merchant payments made by the user.
        # pl.col("TRX_AMT")
        # .filter(pl.col("TRX_TYPE").str.contains("MerchantPay") & (pl.col("TRX_DATETIME")>one_week_ago))
        # .len()
        # .fill_null(0)
        # .alias("merchant_count_last_7D"),

        # # [FEATURE]: p2p_count_last_30D
        # # Total number of peer-to-peer transfers made by the user.
        # pl.col("TRX_AMT")
        # .filter((pl.col("TRX_TYPE") == "P2P") & (pl.col("TRX_DATETIME")>one_month_ago))
        # .len()
        # .fill_null(0)
        # .alias("p2p_count_last_30D"),

        # # [FEATURE]: p2p_count_last_14D
        # # Total number of peer-to-peer transfers made by the user.
        # pl.col("TRX_AMT")
        # .filter((pl.col("TRX_TYPE") == "P2P") & (pl.col("TRX_DATETIME")>two_weeks_ago))
        # .len()
        # .fill_null(0)
        # .alias("p2p_count_last_14D"),

        # # [FEATURE]: p2p_count_last_7D
        # # Total number of peer-to-peer transfers made by the user.
        # pl.col("TRX_AMT")
        # .filter((pl.col("TRX_TYPE") == "P2P") & (pl.col("TRX_DATETIME")>one_week_ago))
        # .len()
        # .fill_null(0)
        # .alias("p2p_count_last_7D"),

        # # [FEATURE]: cash_in_amount_last_30D
        # # Total amount of cash-in transactions made by the user.
        # pl.col("TRX_AMT")
        # .filter((pl.col("TRX_TYPE") == "CashIn") & (pl.col("TRX_DATETIME")>one_month_ago))
        # .sum()
        # .fill_null(0)
        # .alias("cash_in_amount_last_30D"),

        # # [FEATURE]: cash_in_count_last_14D
        # # Total number of cash-in transactions made by the user.
        # pl.col("TRX_AMT")
        # .filter((pl.col("TRX_TYPE") == "CashIn") & (pl.col("TRX_DATETIME")>two_weeks_ago))
        # .len()
        # .fill_null(0)
        # .alias("cash_in_count_last_14D"),

        # # [FEATURE]: cash_in_count_last_7D
        # # Total number of cash-in transactions made by the user.
        # pl.col("TRX_AMT")
        # .filter((pl.col("TRX_TYPE") == "CashIn") & (pl.col("TRX_DATETIME")>one_week_ago))
        # .len()
        # .fill_null(0)
        # .alias("cash_in_count_last_7D"),

        # # [FEATURE]: cash_out_count_last_30D
        # # Total number of cash-out transactions made by the user.
        # pl.col("TRX_AMT")
        # .filter((pl.col("TRX_TYPE") == "CashOut") & (pl.col("TRX_DATETIME")>one_month_ago))
        # .len()
        # .fill_null(0)
        # .alias("cash_out_count_last_30D"),

        # # [FEATURE]: cash_out_count_last_14D
        # # Total number of cash-out transactions made by the user.
        # pl.col("TRX_AMT")
        # .filter((pl.col("TRX_TYPE") == "CashOut") & (pl.col("TRX_DATETIME")>two_weeks_ago))
        # .len()
        # .fill_null(0)
        # .alias("cash_out_count_last_14D"),

        # # [FEATURE]: cash_out_count_last_7D
        # # Total number of cash-out transactions made by the user.
        # pl.col("TRX_AMT")
        # .filter((pl.col("TRX_TYPE") == "CashOut") & (pl.col("TRX_DATETIME")>one_week_ago))
        # .len()
        # .fill_null(0)
        # .alias("cash_out_count_last_7D"),


        # [FEATURE]: bill_pay_ratio
        # Proportion of total transaction volume spent on bills. High values 
        # indicate a utility-centric user who may churn if bill channels fail.
        (
            pl.col("TRX_AMT").filter(pl.col("TRX_TYPE").str.contains("BillPay")).sum().fill_null(0)
            /
            pl.col("TRX_AMT").sum().fill_null(1)
        ).alias("bill_pay_ratio"),

        # [FEATURE]: unique_trx_types_used
        # --------------------------------
        # Counts distinct transaction types (e.g., BillPay, CashIn, MerchantPay).
        # Lower variety suggests low platform integration and higher churn risk.
        pl.col("TRX_TYPE").n_unique().alias("unique_trx_types_used"),

        # [FEATURE]: total_trx_amount
        # ---------------------------
        # Lifetime gross financial volume transacted by the user.
        pl.col("TRX_AMT").sum().alias('total_trx_amount'),

        # [FEATURE]: frequency
        # --------------------
        # Total number of transaction interactions executed by the customer.
        pl.col("TRX_AMT").len().fill_null(0).alias("frequency"),
        
        # [FEATURE]: trx_amount_last_30D
        # ------------------------------
        # Total financial volume transacted specifically during the final month (March 2024).
        pl.col("TRX_AMT")
        .filter(
            (pl.col("TRX_DATETIME").dt.year() == 2024) & 
            (pl.col("TRX_DATETIME").dt.month() == 3)
        )
        .sum().alias('trx_amount_last_30D'),
        
        # [FEATURE]: freq_last_30D
        # -----------------------------
        # Total number of interactions executed during the final month (March 2024).
        pl.col("TRX_AMT")
        .filter(
            (pl.col("TRX_DATETIME").dt.year() == 2024) & 
            (pl.col("TRX_DATETIME").dt.month() == 3)
        )
        .len()
        .fill_null(0)
        .alias("freq_last_30D"),
        
        # [FEATURE]: recency
        # --------------------
        # Days elapsed between the absolute last transaction and the dataset cutoff.
        # High recency values are heavily correlated with silent churn.
        ((dataset_cutoff - pl.col("TRX_DATETIME").max()).dt.total_days()).alias("recency"),

        # [FEATURE]: average_gap_days
        # ---------------------------
        # The typical number of days a user waits between sequential interactions.
        pl.col("TRX_DATETIME")
        .diff()
        .dt.total_days()
        .mean()
        .fill_null(0)
        .alias("average_gap_days"),

        # [FEATURE]: trx_trend
        # --------------------
        # Transaction count momentum (March Count / January Count).
        # Scores < 1.0 indicate a critical slowdown in platform engagement.
        (
            (pl.col("TRX_AMT").filter((pl.col("TRX_DATETIME").dt.year() == 2024) & (pl.col("TRX_DATETIME").dt.month() == 3)).len().fill_null(0) + 1)
            /
            (pl.col("TRX_AMT").filter((pl.col("TRX_DATETIME").dt.year() == 2024) & (pl.col("TRX_DATETIME").dt.month() == 1)).len().fill_null(1) + 1)
        ).alias("trx_trend"),

        # [FEATURE]: consistency_metric
        # -----------------------------
        # Normalized volatility of user habits (Coefficient of Variation of transaction gaps).
        # Higher scores signify erratic, unpredictable usage schedules.
        (
            pl.col("TRX_DATETIME").diff().dt.total_days().std().fill_null(0)
            /
            (pl.col("TRX_DATETIME").diff().dt.total_days().mean().fill_null(1) + 0.1)
        ).alias("consistency_metric"),

        # [FEATURE]: p2p_ratio_90d
        # ------------------------
        # Proportion of user activity dedicated to peer-to-peer transfers.
        # High P2P dominance indicates casual, less sticky use cases.
        (
            pl.col("TRX_AMT").filter((pl.col("TRX_TYPE")=="P2P")).sum().fill_null(0)
            /
            pl.col("TRX_AMT").sum().fill_null(1)
        ).alias("p2p_ratio_90d"),

        # [FEATURE]: importance
        # ---------------------
        # Financial weight of the customer relative to the platform's average user volume.
        # Helps business logic isolate and prioritize "Whale" accounts over casual ones.
        (pl.col("TRX_AMT").sum() / avg_total_volume_per_user)
        .fill_null(0)
        .alias("importance"),
        
        # [FEATURE]: freq_last_14D
        # ------------------------
        # Absolute transaction frequency count inside the final two-week window.
        pl.col("TRX_AMT").filter(pl.col("TRX_DATETIME") >= two_weeks_ago).len().fill_null(0).alias("freq_last_14D"),

        # [FEATURE]: freq_last_7D
        # ------------------------
        # Absolute transaction frequency count inside the final two-week window.
        pl.col("TRX_AMT").filter(pl.col("TRX_DATETIME") >= one_week_ago).len().fill_null(0).alias("freq_last_7D"),
    ])
    .with_columns([
        # [FEATURE]: velocity_last_14d
        # ----------------------------
        # The share of overall lifetime interactions that occurred in the last fortnight.
        # Near-zero values signify a severe structural drop-off in active habit.
        (pl.col("freq_last_14D") / (pl.col("frequency") + 1)).alias("velocity_last_14d"),

        # [FEATURE]: velocity_last_7d
        # ----------------------------
        # The share of overall lifetime interactions that occurred in the last fortnight.
        # Near-zero values signify a severe structural drop-off in active habit.
        (pl.col("freq_last_7D") / (pl.col("frequency") + 1)).alias("velocity_last_7d"),
        
        # [FEATURE]: inactivity_multiplier
        # --------------------------------
        # Measures how many times longer a user has been dormant relative to their typical rhythm.
        # Scores > 3.0 mean the user has missed multiple expected usage windows.
        (pl.col("recency") / (pl.col("average_gap_days") + 0.1)).alias("inactivity_multiplier")
    ])
    .rename({"SRC_ACCOUNT": "ACCOUNT_ID"})
)


## Feature set 3 — Wallet liquidity profile metrics

**Hypothesis:** `balance_std_dev` near zero on a flat low balance can signal an
abandoned account. `balance_trend` near zero signals active cash drain — the user
withdrawing funds before leaving the platform entirely.

In [4]:
# ==============================================================================
# FEATURE SET 4: WALLET LIQUIDITY PROFILE METRICS
# ==============================================================================
balance_features = (
    lazy_balances
    .with_columns(pl.col("DATE").str.to_date("%Y-%m-%d"))
    .group_by("ACCOUNT_ID")
    .agg([
        # [FEATURE]: balance_std_dev
        # --------------------------
        # Measures fluctuations in wallet funds. Highly stable balances without deviations
        # can signify a stale or abandoned account that has been emptied out.
        pl.col("AVAILABLE_BALANCE")
        .filter((pl.col("DATE").dt.year() == 2024) & (pl.col("DATE").dt.month() == 3))
        .std()
        .fill_null(0)
        .alias("balance_std_dev_last30D"),

        pl.col("AVAILABLE_BALANCE")
        .filter(pl.col("DATE")>two_weeks_ago)
        .std()
        .fill_null(0)
        .alias("balance_std_dev_last14D"),

        pl.col("AVAILABLE_BALANCE")
        .filter(pl.col("DATE")>one_week_ago)
        .std()
        .fill_null(0)
        .alias("balance_std_dev_last7D"),

        # [FEATURE]: balance_trend
        # ------------------------
        # Financial health velocity (Average March Balance / Average Lifetime Balance).
        # Values turning close to 0 specify cash drain, indicating the user is withdrawing 
        # their funds prior to abandoning the wallet.
        (
            (pl.col("AVAILABLE_BALANCE").filter((pl.col("DATE").dt.year() == 2024) & (pl.col("DATE").dt.month() == 3)).mean().fill_null(0) + 1)
            /
            (pl.col("AVAILABLE_BALANCE").mean().fill_null(1) + 1)
        ).alias("balance_trend")
    ])
)

# Preview execution slice
# print(balance_features.fetch(10))

## Building the pipeline — joining all feature sets

All four feature groups are joined on `ACCOUNT_ID`. Null-fills here represent the
"no activity" case for each feature (e.g. a user with zero transactions gets
`frequency = 0`, not a null).

In [5]:
final_pipe = (
    balance_features
    .join(trx_features, on=USER_COL, how="left")
    # .join(kyc_features, on=USER_COL, how='left')
    .with_columns([
        # pl.col("balance_std_dev").fill_null(0),
        pl.col("frequency").fill_null(0),
        pl.col("recency").fill_null(99),              # Use a standard flag for inactive users
        pl.col("average_gap_days").fill_null(-0.5),
        pl.col("trx_amount_last_30D").fill_null(0),
        pl.col("trx_trend").fill_null(1),
        pl.col("consistency_metric").fill_null(0),
        pl.col("importance").fill_null(0),
        pl.col("balance_trend").fill_null(1),
    ])
)

In [ ]:
print("Streaming datasets and calculating features...")
features_df = final_pipe.collect(streaming=True)
feature_columns = features_df.columns[1:]
feature_columns

Streaming datasets and calculating features...


## Save output for downstream notebooks

This is the key step that makes notebooks independent — `03_feature_quality.ipynb`
loads this file directly instead of re-running all the aggregations above.

In [ ]:
features_df.write_parquet(f"exp_files/features.parquet")

## Next notebook

Continue to **`03_feature_quality.ipynb`**, which loads `features.parquet` and
handles skewness, outliers, and zero-inflation.